# Paper Final Benchmark (Colab H100)

This notebook is for your paper-grade benchmark run.

It does four things:
1. Mounts Google Drive and enters the repo.
2. Creates a run-specific config from `configs/paper_final_500.toml`.
3. Builds graphs (if needed).
4. Runs 500-epoch benchmarking across `ff_layerwise`, `ff_e2e`, and `backprop`, then writes a paper summary table.
Implementation checks in this notebook:
- verifies benchmark outputs include fold stability columns (mean/std/min) for economics
- validates critic-aware FF benchmark metadata columns before generating paper summary


In [1]:
from pathlib import Path
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

ROOT = Path("/content/drive/MyDrive/Forward-Risk-Manager") if IN_COLAB else Path.cwd()
if not ROOT.exists():
    raise FileNotFoundError(
        f"Repo root not found at {ROOT}. Update ROOT in this cell to your Drive path."
    )
os.chdir(ROOT)
print("repo root:", ROOT)
print("cwd:", Path.cwd())

Mounted at /content/drive
repo root: /content/drive/MyDrive/Forward-Risk-Manager
cwd: /content/drive/MyDrive/Forward-Risk-Manager


In [2]:
import os
import re
import shlex
import subprocess
import time
from collections import deque

def run(cmd: str, allow_fail: bool = False, tail_lines: int = 200) -> bool:
    print("\n" + "=" * 120)
    print(cmd)
    print("=" * 120)
    t0 = time.time()

    env = os.environ.copy()
    env.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

    logs_dir = ROOT / "runs" / "experiments" / "_paper_logs"
    logs_dir.mkdir(parents=True, exist_ok=True)
    safe_name = re.sub(r"[^A-Za-z0-9._-]+", "_", cmd).strip("_")[:120] or "command"
    log_path = logs_dir / f"{int(time.time())}_{safe_name}.log"

    proc = subprocess.Popen(
        cmd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert proc.stdout is not None

    tail = deque(maxlen=max(40, int(tail_lines)))
    with log_path.open("w", encoding="utf-8") as lf:
        for line in proc.stdout:
            print(line, end="")
            lf.write(line)
            tail.append(line.rstrip("\n"))
    rc = proc.wait()

    elapsed = time.time() - t0
    print(f"\ncompleted in {elapsed:.2f}s | log: {log_path}")
    if rc != 0:
        if tail:
            print("---- command output tail ----")
            for ln in tail:
                print(ln)
            print("---- end tail ----")
        msg = f"command failed ({rc}): {cmd}"
        if allow_fail:
            print("WARNING:", msg)
            return False
        raise RuntimeError(msg)
    return True

INSTALL_DEPS = True
if INSTALL_DEPS:
    run("python -m pip install --upgrade pip")
    run("python -m pip install -r requirements.txt")
    run("python -m pip install -e .")
else:
    print("Skipping dependency install (INSTALL_DEPS=False)")


python -m pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 85.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2

completed in 4.71s | log: /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/_paper_logs/1771382748_python_-m_pip_install_--upgrade_pip.log

python -m pip install -r requirements.txt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 57.1 MB/s  0:00:00

completed in 2.81s | log: /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/_paper_logs/1771382751_python_-m_pip_install_-r_requirements.txt.log

python -m pip install -e .
Obtaining file:///content/drive/MyDrive/Forward-Risk-Manager
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with

In [3]:
from datetime import datetime, timezone

TEMPLATE_CONFIG = ROOT / "configs" / "paper_final_500.toml"
assert TEMPLATE_CONFIG.exists(), f"Missing template config: {TEMPLATE_CONFIG}"

BASE_TOKEN = "runs/experiments/paper_final_500_TEMPLATE"
RUN_ID_OVERRIDE = "paper_final_500_20260218_024603"  # e.g. "paper_final_500_20260218_024603" to resume an existing run

if RUN_ID_OVERRIDE.strip():
    RUN_ID = RUN_ID_OVERRIDE.strip()
    RUN_ROOT = ROOT / "runs" / "experiments" / RUN_ID
    runtime_config = RUN_ROOT / "runtime_config.toml"
    assert runtime_config.exists(), f"Missing runtime config for resume: {runtime_config}"
    print("resuming run id:", RUN_ID)
    print("runtime config:", runtime_config)
    print("run root:", RUN_ROOT)
else:
    RUN_ID = f"paper_final_500_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}"
    RUN_ROOT = ROOT / "runs" / "experiments" / RUN_ID

    for sub in ("data", "metrics", "plots", "logs", "models", "configs"):
        (RUN_ROOT / sub).mkdir(parents=True, exist_ok=True)

    runtime_config = RUN_ROOT / "runtime_config.toml"
    cfg_txt = TEMPLATE_CONFIG.read_text()
    assert BASE_TOKEN in cfg_txt, f"Expected token {BASE_TOKEN!r} in template config"
    cfg_txt = cfg_txt.replace(BASE_TOKEN, f"runs/experiments/{RUN_ID}")
    runtime_config.write_text(cfg_txt)

    print("run id:", RUN_ID)
    print("runtime config:", runtime_config)
    print("run root:", RUN_ROOT)


run id: paper_final_500_20260218_024603
runtime config: /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260218_024603/runtime_config.toml
run root: /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260218_024603


In [4]:
graphs_path = RUN_ROOT / "data" / "graphs.pt"
RUN_BUILD_GRAPHS = not graphs_path.exists()
FORCE_REBUILD_GRAPHS = False

if FORCE_REBUILD_GRAPHS:
    RUN_BUILD_GRAPHS = True

if RUN_BUILD_GRAPHS:
    run(
        f"python -u scripts/build_graphs.py --config {shlex.quote(str(runtime_config))}"
    )
else:
    print(f"Skipping graph build; existing graphs found at {graphs_path}")



python -u scripts/build_graphs.py --config /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260218_024603/runtime_config.toml
Macro features: source=auto cols=12 rows=6546

Building graphs: 100%|██████████| 6527/6527 [01:47<00:00, 60.58win/s]
Wrote runs/experiments/paper_final_500_20260218_024603/data/graphs.pt with 2064 graphs
Date range: 2011-06-01 -> 2026-02-06 | windows: 6527 | built: 2064 | skipped_lag=0, skipped: members=4463, cols=0, min_nodes=0, no_edges=0

completed in 147.10s | log: /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/_paper_logs/1771382764_python_-u_scripts_build_graphs.py_--config_content_drive_MyDrive_Forward-Risk-Manager_runs_experiments_paper_final_500_2.log


In [ ]:
import csv
import re

BENCH_MODES = ["ff_layerwise", "ff_e2e", "backprop"]
RESUME_BENCHMARK = True
FORCE_RERUN_MODES = set()  # e.g. {"backprop"}

benchmark_csv = RUN_ROOT / "metrics" / "benchmark.csv"
folds_csv = RUN_ROOT / "metrics" / "benchmark_walk_forward_folds.csv"

per_mode_benchmark = {
    mode: RUN_ROOT / "metrics" / f"benchmark_{mode}.csv" for mode in BENCH_MODES
}
per_mode_folds = {
    mode: RUN_ROOT / "metrics" / f"benchmark_walk_forward_folds_{mode}.csv" for mode in BENCH_MODES
}

cfg_template = runtime_config.read_text()
assert "out_csv" in cfg_template and "walk_forward_out_csv" in cfg_template, (
    "runtime config missing benchmark out paths"
)

(RUN_ROOT / "configs").mkdir(parents=True, exist_ok=True)

for mode in BENCH_MODES:
    mode_csv = per_mode_benchmark[mode]
    mode_folds = per_mode_folds[mode]

    if (
        RESUME_BENCHMARK
        and mode not in FORCE_RERUN_MODES
        and mode_csv.exists()
        and mode_csv.stat().st_size > 0
    ):
        print(f"Skipping mode={mode}; found {mode_csv}")
        continue

    cfg_mode = cfg_template
    cfg_mode = re.sub(
        r'(?m)^out_csv\s*=\s*".*"\s*$',
        f'out_csv = "{mode_csv.as_posix()}"',
        cfg_mode,
    )
    cfg_mode = re.sub(
        r'(?m)^walk_forward_out_csv\s*=\s*".*"\s*$',
        f'walk_forward_out_csv = "{mode_folds.as_posix()}"',
        cfg_mode,
    )

    cfg_mode_path = RUN_ROOT / "configs" / f"runtime_config_{mode}.toml"
    cfg_mode_path.write_text(cfg_mode)

    run(
        f"python -u scripts/benchmark_training.py "
        f"--config {shlex.quote(str(cfg_mode_path))} "
        f"--modes {mode}"
    )


def combine_csv(paths, out_path):
    rows = []
    fieldnames = []
    for p in paths:
        if not p.exists() or p.stat().st_size == 0:
            continue
        with p.open("r", newline="") as f:
            reader = csv.DictReader(f)
            if reader.fieldnames:
                for k in reader.fieldnames:
                    if k not in fieldnames:
                        fieldnames.append(k)
            for row in reader:
                rows.append(row)

    if not rows:
        raise RuntimeError(f"No rows found to combine into {out_path}")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)

    print(f"Wrote combined CSV: {out_path} rows={len(rows)}")


combine_csv([per_mode_benchmark[m] for m in BENCH_MODES], benchmark_csv)
combine_csv([per_mode_folds[m] for m in BENCH_MODES], folds_csv)



python -u scripts/benchmark_training.py --config /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260218_024603/runtime_config.toml --modes ff_layerwise,ff_e2e,backprop
econ ticker: requested=AUTO effective=BWC source=auto_max_rows rows=10439
risk ticker: requested=AUTO effective=BWC source=auto_max_rows rows=10439 horizons=[21]
portfolio ticker: requested=AUTO effective=BWC source=auto_max_rows rows=10439 horizon=21
walk-forward splits=3 (train_frac=0.6, eval_frac=0.2, step_frac=0.1)
risk_head disabled for ff_layerwise benchmarking.
portfolio_head disabled for ff_layerwise benchmarking.

Benchmark: 100%|██████████| 500/500 [14:08<00:00,  1.70s/epoch]
calibrated goodness_target=10.8347 (train-cal acc=0.8316)
risk_head disabled for ff_layerwise benchmarking.
portfolio_head disabled for ff_layerwise benchmarking.

Benchmark: 100%|██████████| 500/500 [16:38<00:00,  2.00s/epoch]
calibrated goodness_target=38.8234 (train-cal acc=0.4841)
risk_head disabled for 

In [6]:
benchmark_csv = RUN_ROOT / "metrics" / "benchmark.csv"
folds_csv = RUN_ROOT / "metrics" / "benchmark_walk_forward_folds.csv"
summary_md = RUN_ROOT / "logs" / "paper_benchmark_summary.md"
summary_csv = RUN_ROOT / "metrics" / "paper_benchmark_summary.csv"
summary_json = RUN_ROOT / "logs" / "paper_benchmark_summary.json"

import pandas as pd

assert benchmark_csv.exists(), f"Missing benchmark CSV: {benchmark_csv}"
bench_df = pd.read_csv(benchmark_csv)
required_cols = [
    "mode",
    "row_type",
    "objective_track",
    "econ_sharpe_uplift",
    "econ_sharpe_uplift_std",
    "econ_sharpe_uplift_min",
    "risk_head_enabled_effective",
    "portfolio_head_enabled_effective",
    "econ_regime_gate_enabled",
    "econ_regime_confidence_mean",
    "econ_regime_exposure_mean",
]
missing = [c for c in required_cols if c not in bench_df.columns]
assert not missing, f"Benchmark CSV missing expected columns: {missing}"
print("benchmark column check passed:", required_cols)

run(
    f"python -u scripts/paper_benchmark_summary.py "
    f"--benchmark {shlex.quote(str(benchmark_csv))} "
    f"--folds {shlex.quote(str(folds_csv))} "
    f"--out-md {shlex.quote(str(summary_md))} "
    f"--out-csv {shlex.quote(str(summary_csv))} "
    f"--out-json {shlex.quote(str(summary_json))}"
)

print("benchmark:", benchmark_csv)
print("folds:", folds_csv)
print("summary md:", summary_md)
print("summary csv:", summary_csv)
print("summary json:", summary_json)



python -u scripts/paper_benchmark_summary.py --benchmark /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260215_180439/metrics/benchmark.csv --folds /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260215_180439/metrics/benchmark_walk_forward_folds.csv --out-md /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260215_180439/logs/paper_benchmark_summary.md --out-csv /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260215_180439/metrics/paper_benchmark_summary.csv --out-json /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260215_180439/logs/paper_benchmark_summary.json
Paper Benchmark Summary
source: /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260215_180439/metrics/benchmark.csv
| mode | quality_metric | quality_value | eval_auroc | eval_auprc | avg_epoch_s | graphs_per_s | econ_sharpe_uplift | econ_ann_

In [7]:
from IPython.display import Markdown, display

if summary_md.exists():
    display(Markdown(summary_md.read_text()))
else:
    print("Missing summary markdown:", summary_md)

print("\nArtifact check:")
for path in [benchmark_csv, folds_csv, summary_md, summary_csv, summary_json]:
    print(f"{path} -> exists={path.exists()} size={path.stat().st_size if path.exists() else 0}")

# Paper Benchmark Summary

Source: `/content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260215_180439/metrics/benchmark.csv`

| mode | quality_metric | quality_value | eval_auroc | eval_auprc | avg_epoch_s | graphs_per_s | econ_sharpe_uplift | econ_ann_return_uplift |
|---|---|---|---|---|---|---|---|---|
| ff_layerwise | eval_sep | 9.3272 | 0.9761 | 0.9841 | 0.609 | 2372.2 | 0.013 | -12.36% |
| ff_e2e | eval_sep | 24.2317 | 1.0000 | 1.0000 | 0.418 | 3447.8 | 0.016 | -12.89% |
| backprop | eval_auroc | 1.0000 | 1.0000 | 1.0000 | 0.359 | 4027.6 | -0.405 | -21.73% |

## Key Points
- Fastest: `backprop` (4027.6 graphs/s).
- Best quality: `ff_e2e` (eval_sep=24.2317).
- Best economics: `ff_e2e` (Sharpe uplift=0.016, ann return uplift=-12.89%).



Artifact check:
/content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260215_180439/metrics/benchmark.csv -> exists=True size=9851
/content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260215_180439/metrics/benchmark_walk_forward_folds.csv -> exists=True size=12171
/content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260215_180439/logs/paper_benchmark_summary.md -> exists=True size=776
/content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260215_180439/metrics/paper_benchmark_summary.csv -> exists=True size=1235
/content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260215_180439/logs/paper_benchmark_summary.json -> exists=True size=7141
